In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# MLSecOps — Crack Segmentation Training\n", "Automated CT pipeline — do not edit manually"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "\n",
    "# ── AWS credentials (injected via Kaggle secrets) ──────────────────────────\n",
    "os.environ['AWS_ACCESS_KEY_ID']     = os.environ.get('CT_AWS_ACCESS_KEY_ID', '')\n",
    "os.environ['AWS_SECRET_ACCESS_KEY'] = os.environ.get('CT_AWS_SECRET_ACCESS_KEY', '')\n",
    "os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'\n",
    "\n",
    "print('AWS credentials configured')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Install dependencies ───────────────────────────────────────────────────\n",
    "!pip install -q ultralytics mlflow boto3 'dvc[s3]'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Copy code from Kaggle dataset input ───────────────────────────────────\n",
    "import shutil\n",
    "import os\n",
    "\n",
    "# Kaggle mounts datasets at /kaggle/input/<dataset-name>/\n",
    "SOURCE = '/kaggle/input/mlsecops-crack-seg-code'\n",
    "DEST   = '/kaggle/working'\n",
    "\n",
    "for item in os.listdir(SOURCE):\n",
    "    src = os.path.join(SOURCE, item)\n",
    "    dst = os.path.join(DEST, item)\n",
    "    if os.path.isdir(src):\n",
    "        shutil.copytree(src, dst, dirs_exist_ok=True)\n",
    "    else:\n",
    "        shutil.copy2(src, dst)\n",
    "\n",
    "os.chdir(DEST)\n",
    "print('Working directory:', os.getcwd())\n",
    "print('Files:', os.listdir('.'))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Configure DVC S3 remote ────────────────────────────────────────────────\n",
    "!dvc remote modify s3_remote access_key_id $CT_AWS_ACCESS_KEY_ID\n",
    "!dvc remote modify s3_remote secret_access_key $CT_AWS_SECRET_ACCESS_KEY"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Run training pipeline ──────────────────────────────────────────────────\n",
    "!python train.py \\\n",
    "    --epochs 5 \\\n",
    "    --imgsz 640 \\\n",
    "    --batch 16 \\\n",
    "    --workers 2"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ── Push model to S3 ──────────────────────────────────────────────────────\n",
    "import glob\n",
    "import boto3\n",
    "import os\n",
    "\n",
    "GIT_SHA       = os.environ.get('GIT_SHA', 'unknown')\n",
    "MODELS_BUCKET = 'mlsecops-models-351611731527'\n",
    "\n",
    "model_files = glob.glob('runs/segment/crack_seg/weights/crack_seg_*.pt')\n",
    "if not model_files:\n",
    "    raise FileNotFoundError('No model file found after training')\n",
    "\n",
    "model_path = model_files[0]\n",
    "model_name = os.path.basename(model_path)\n",
    "s3_key     = f'crack-seg/{GIT_SHA}/{model_name}'\n",
    "\n",
    "s3 = boto3.client('s3', region_name='us-east-1')\n",
    "s3.upload_file(model_path, MODELS_BUCKET, s3_key)\n",
    "\n",
    "print(f'Model pushed to: s3://{MODELS_BUCKET}/{s3_key}')"
   ]
  }
 ]
}